In [8]:
import torch
torch.cuda.is_available()

True

In [7]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


class SentimentTrainer:
    """
    Training pipeline for sentiment classification.
    """

    def __init__(self, model_name="distilbert-base-uncased", model_dir="model"):
        """
        Initialize tokenizer, model, and dataset
        """
        self.model_name = model_name
        self.model_dir = model_dir
        
        print("Loading dataset...")
        self.dataset = load_dataset("imdb")

        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        print("Loading pretrained model...")
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2
        )

    def tokenize(self, examples):
        """
        Convert text into tokens that the model can understand
        """
        return self.tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=256
        )

    def prepare_dataset(self):
        """
        Apply tokenization to the dataset
        """
        print("Tokenizing dataset...")
        self.tokenized_dataset = self.dataset.map(self.tokenize, batched=True)

    def compute_metrics(self, eval_pred):
        """
        Compute evaluation metrics for the model
        """
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=1)

        precision, recall, f1, _ = precision_recall_fscore_support(
            labels,
            predictions,
            average="binary"
        )

        accuracy = accuracy_score(labels, predictions)

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

    def train(self):
        """
        Train the model
        """
        training_args = TrainingArguments(
            output_dir=self.model_dir,
            learning_rate=2e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=2,
            weight_decay=0.01,
            save_strategy="epoch"
        )

        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.tokenized_dataset["train"],
            eval_dataset=self.tokenized_dataset["test"],
            processing_class=self.tokenizer,
            compute_metrics=self.compute_metrics
        )

        print("Starting training...")
        self.trainer.train()

    def evaluate(self):
        """
        Evaluate model performance
        """
        print("Evaluating model...")
        results = self.trainer.evaluate()

        print("\nEvaluation Results")
        print(results)

    def save_model(self):
        """
        Save trained model and tokenizer
        """
        print("Saving model...")
        self.trainer.save_model(self.model_dir)
        self.tokenizer.save_pretrained(self.model_dir)

        print("Model saved to 'sentiment_model'")

In [ ]:
trainer = SentimentTrainer()

In [ ]:
trainer.prepare_dataset()

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model()